In [48]:
import os
import yaml
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# **Load Config**

In [49]:
def load_config(config_path):
    """
    Load the configuration file.

    Parameters:
    ----------
    config_path : str
        Configuration file location.
    
    Returns:
    -------
    config : dict
        Loaded configuration file.
    """

    try:
        # load config.yaml
        with open(config_path, 'r') as file:
            config = yaml.safe_load(file)
        # .yaml is empty
        if config is None:
            raise ValueError('Configuration file is empty')

    # yaml is not found in directory
    except FileNotFoundError:
        raise FileNotFoundError(f'Configuration file not found in {config_path}')
    
    # yaml syntax error
    except yaml.YAMLError as e:
        raise ValueError('Invalid YAML format') from e

    return config

# **Update Config**

In [50]:
def update_config(key, value, config, path_config):
    # add or update the configuration params
    config[key] = value

    # open config file
    with open(path_config, 'w') as file:
        yaml.dump(config, file)
    
    print(f'Config params updated! \nKey: {key} \nValue: {value}')

    # reload config
    config = load_config(config_path=path_config)
    return config

In [51]:
# load config
loaded_config = load_config(config_path='../config/config.yaml')
loaded_config

{'co_range': [-1, 47],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'no2_range': [-1, 65],
 'o3_range': [-1, 151],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'pm10_range': [-1, 179],
 'pm25_range': [-1, 174],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat'],
 'raw_data_path': '../data/raw/',
 'so2_range': [-1, 82]}

In [52]:
# check the entire file on raw data path
os.listdir(loaded_config['raw_data_path'])

['indeks-standar-pencemar-udara-di-spku-bulan-desember-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-juli-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-oktober-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-agustus-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-september-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-april-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-februari-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-juni-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-november-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-januari-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-mei-tahun-2021.csv',
 'indeks-standar-pencemar-udara-di-spku-bulan-maret-tahun-2021.csv']

# **2. Data Collection**

In [53]:
def load_data(path_data):

    # make an empty dataframe
    raw_dataset = pd.DataFrame()

    for i in os.listdir(path_data):
        raw_dataset = pd.concat([pd.read_csv(path_data+i), raw_dataset])
    
    return raw_dataset

In [54]:
data = load_data(path_data=loaded_config['raw_data_path']).reset_index(drop=True)
data

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-03-01,DKI1 (Bunderan HI),37,60,20,12,12,17,60,PM25,SEDANG
1,2021-03-02,DKI1 (Bunderan HI),38,55,20,15,14,28,55,PM25,SEDANG
2,2021-03-03,DKI1 (Bunderan HI),49,67,19,21,17,35,67,PM25,SEDANG
3,2021-03-04,DKI1 (Bunderan HI),51,75,22,18,19,31,75,PM25,SEDANG
4,2021-03-05,DKI1 (Bunderan HI),52,69,20,16,16,30,69,PM25,SEDANG
...,...,...,...,...,...,...,...,...,...,...,...
1825,2021-12-27,DKI5 (Kebon Jeruk) Jakarta Barat,54,76,36,14,21,47,76,PM25,SEDANG
1826,2021-12-28,DKI5 (Kebon Jeruk) Jakarta Barat,44,68,20,11,21,33,68,PM25,SEDANG
1827,2021-12-29,DKI5 (Kebon Jeruk) Jakarta Barat,34,54,28,8,25,29,54,PM25,SEDANG
1828,2021-12-30,DKI5 (Kebon Jeruk) Jakarta Barat,53,75,25,15,23,44,75,PM25,SEDANG


In [55]:
# add interim data path to the config file
PATH_JOINED_DATA = '../data/interim/joined_dataset.pkl'

# update config
config = update_config(
    key='path_joined_data',
    value=PATH_JOINED_DATA,
    config=loaded_config,
    path_config='../config/config.yaml'
)

Config params updated! 
Key: path_joined_data 
Value: ../data/interim/joined_dataset.pkl


In [56]:
# reload config
config

{'co_range': [-1, 47],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'no2_range': [-1, 65],
 'o3_range': [-1, 151],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'pm10_range': [-1, 179],
 'pm25_range': [-1, 174],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat'],
 'raw_data_path': '../data/raw/',
 'so2_range': [-1, 82]}

# **3. Data Validation**

In [57]:
data.dtypes

tanggal        str
stasiun        str
pm10        object
pm25        object
so2         object
co          object
o3          object
no2         object
max         object
critical       str
categori       str
dtype: object

## Handling column tanggal

In [58]:
data['tanggal'] = pd.to_datetime(data['tanggal'])
data.dtypes

tanggal     datetime64[us]
stasiun                str
pm10                object
pm25                object
so2                 object
co                  object
o3                  object
no2                 object
max                 object
critical               str
categori               str
dtype: object

## Handling column pm10

In [59]:
data['pm10'].unique()

array(['37', '38', '49', '51', '52', '57', '47', '36', '45', '27', '44',
       '39', '55', '59', '64', '60', '43', '32', '25', '50', '53', '54',
       '48', '56', '65', '71', '40', '67', '78', '73', '61', '33', '62',
       '41', '29', '66', '81', '46', '58', '24', '28', '68', '30', '35',
       '---', '26', '23', '70', '74', '69', '86', '76', '42', '75', '72',
       '63', '34', '31', '20', '21', '22', '15', '84', '89', '14', '18',
       '19', '80', '16', '17', '94', '77', '83', '92', '79', '85', '82',
       '90', '87', '95', '88', 51, 47, 50, 52, 56, 55, 53, 32, 65, 59, 60,
       58, 62, 79, 64, 61, 76, 63, 36, 69, 72, 73, 71, 68, 74, 48, 45, 57,
       54, 66, 67, 87, 42, 35, 44, 46, 49, 26, 83, '100', '179'],
      dtype=object)

In [60]:
# change --- to -1
# ensure there is no -1 in data
data.eq(-1).any()

tanggal     False
stasiun     False
pm10        False
pm25        False
so2         False
co          False
o3          False
no2         False
max         False
critical    False
categori    False
dtype: bool

In [61]:
# replace --- with -1
data['pm10'] = data['pm10'].replace('---', '-1').astype(int)
data['pm10'].unique()

array([ 37,  38,  49,  51,  52,  57,  47,  36,  45,  27,  44,  39,  55,
        59,  64,  60,  43,  32,  25,  50,  53,  54,  48,  56,  65,  71,
        40,  67,  78,  73,  61,  33,  62,  41,  29,  66,  81,  46,  58,
        24,  28,  68,  30,  35,  -1,  26,  23,  70,  74,  69,  86,  76,
        42,  75,  72,  63,  34,  31,  20,  21,  22,  15,  84,  89,  14,
        18,  19,  80,  16,  17,  94,  77,  83,  92,  79,  85,  82,  90,
        87,  95,  88, 100, 179])

## Handling column pm25

In [62]:
# check pm25
data['pm25'].unique()

array(['60', '55', '67', '75', '69', '73', '88', '63', '50', '65', '35',
       '58', '62', '70', '80', '93', '102', '61', '54', '53', '47', '71',
       '77', '57', '66', '76', '79', '81', '89', '84', '72', '107', '115',
       '94', '83', '68', '97', '52', '99', '106', '87', '118', '64', '19',
       '24', '22', '96', '---', '103', '95', '101', '123', '125', '91',
       '98', '85', '78', '41', '92', '117', '59', '82', '110', '108',
       '74', '112', '90', '116', '86', '23', '21', '113', '105', '56',
       '109', '46', '48', '25', '44', '34', '51', '20', '37', '49', '39',
       '42', '45', '29', '33', '126', '16', '13', nan, '40', '30', '28',
       '31', '100', '38', '43', '36', '26', '129', '111', '120', '104',
       '119', '114', '124', '132', '121', '122', '141', '161', '153',
       '130', '145', '156', '174', '138', '134', '150', '140', '147',
       '148', '131', '154', '27', 68, 63, 70, 66, 73, 71, 69, 60, 50, 75,
       92, 72, 77, 78, 89, 84, 80, 81, 87, 93, 96, 112, 8

In [63]:
# replace with -1
data['pm25'] = data['pm25'].fillna(-1)
data['pm25'] = data['pm25'].replace('---', -1).astype(int)
print('pm25 data type:', data['pm25'].dtype)
data['pm25'].unique()

pm25 data type: int64


array([ 60,  55,  67,  75,  69,  73,  88,  63,  50,  65,  35,  58,  62,
        70,  80,  93, 102,  61,  54,  53,  47,  71,  77,  57,  66,  76,
        79,  81,  89,  84,  72, 107, 115,  94,  83,  68,  97,  52,  99,
       106,  87, 118,  64,  19,  24,  22,  96,  -1, 103,  95, 101, 123,
       125,  91,  98,  85,  78,  41,  92, 117,  59,  82, 110, 108,  74,
       112,  90, 116,  86,  23,  21, 113, 105,  56, 109,  46,  48,  25,
        44,  34,  51,  20,  37,  49,  39,  42,  45,  29,  33, 126,  16,
        13,  40,  30,  28,  31, 100,  38,  43,  36,  26, 129, 111, 120,
       104, 119, 114, 124, 132, 121, 122, 141, 161, 153, 130, 145, 156,
       174, 138, 134, 150, 140, 147, 148, 131, 154,  27, 127, 157, 136])

## Handling column so2

In [64]:
data['so2'].unique()

array(['20', '19', '22', '23', '21', '24', '27', '29', '32', '25', '30',
       '28', '45', '50', '46', '54', '59', '42', '26', '33', '41', '55',
       '43', '39', '35', '49', '48', '34', '31', '40', '12', '44', '---',
       '38', 23, 22, 26, 31, 19, 20, 18, 24, 21, 27, 29, 51, 47, 50, 54,
       53, 61, 46, 52, 49, 55, 48, 58, 62, 64, 74, 13, 12, 17, 43, 37, 45,
       9, 38, 39, 41, 36, 32, 57, 40, 34, 25, '16', '17', '15', '14',
       '13', '2', '11', '18', '37', '36', '77', '7', '5', '8', '4', '3',
       '51', '52', '47', '53', '56', '62', '9', '63', '61', '60', '57',
       '10', 28, 33, 30, 44, 42, 10, 35, '67', '66', '64', '76', '65',
       '58', '73', '82', '81', '80'], dtype=object)

In [65]:
data['so2'] = data['so2'].replace('---', -1).astype(int)
print('so2 data type:', data['so2'].dtypes)

so2 data type: int64


## Handling column co

In [66]:
data['co'].unique()

array(['12', '15', '21', '18', '16', '22', '23', '20', '9', '13', '14',
       '19', '11', '10', '17', '7', '8', '6', '---', '25', '5', '4', '26',
       '3', '31', '27', '43', '29', '28', '32', '33', '42', '38', '34',
       '47', '44', '2', '24', '30', 8, 10, 11, 9, 15, 7, 6, 13, 14, 12,
       17, 16, 19, 20, 4], dtype=object)

In [67]:
data['co'] = data['co'].replace('---', -1).astype(int)
print('co data type:', data['co'].dtypes)

co data type: int64


## Handling column no

In [68]:
data['no2'].unique()

array(['17', '28', '35', '31', '30', '37', '19', '22', '27', '21', '25',
       '34', '36', '45', '40', '18', '16', '15', '33', '23', '26', '9',
       '41', '24', '14', '11', '8', '13', '20', '10', '6', '12', '7',
       '---', '42', '39', '32', '5', '38', '29', '4', '1', '3', '47',
       '46', '44', '51', 22, 28, 35, 26, 27, 18, 16, 15, 25, 19, 36, 20,
       34, 30, 33, 31, 29, 41, 13, 14, 12, 10, 9, 17, 21, 24, 11, 32, 23,
       52, 40, '48', '43', '49', '57', '65', '52', '62', '53'],
      dtype=object)

In [69]:
data['no2'] = data['no2'].replace('---', -1).astype(int)
print('no2 data type:', data['no2'].dtypes)

no2 data type: int64


## Handling column o3

In [70]:
data['o3'].unique()

array(['12', '14', '17', '19', '16', '15', '11', '10', '13', '21', '20',
       '25', '24', '18', '9', '31', '38', '53', '50', '49', '59', '52',
       '51', '41', '33', '42', '43', '45', '37', '57', '74', '56', '44',
       '63', '71', '67', '55', '48', '28', '39', '34', '29', '23', '36',
       '27', '32', '22', '35', '---', '26', '66', '46', '54', '61', '58',
       '40', '65', '72', '64', '62', '60', '151', '47', '30', '80', '86',
       '77', '85', '68', '93', '78', '104', '75', '134', '8', '73', '81',
       29, 25, 19, 24, 21, 11, 18, 23, 20, 17, 26, 27, 32, 34, 22, 40, 30,
       39, 35, 67, 59, 47, 50, 58, 38, 48, 57, 55, 49, 53, 46, 42, 33, 41,
       31, 13, 15, 28, 36, 12, 44, '70', '7'], dtype=object)

In [71]:
data['o3'] = data['o3'].replace('---', -1).astype(int)
print('o3 data type:', data['o3'].dtypes)

o3 data type: int64


## Handling column max

In [72]:
data['max'].unique()

array([60, 55, 67, 75, 69, 73, 88, 63, 50, 65, 35, 58, 62, 70, 80, 93,
       102, 61, 54, 53, 47, 71, 77, 57, 66, 76, 79, 81, 89, 84, 72, 107,
       115, 94, 83, 68, 97, 52, 99, 106, 87, 118, 64, 51, 59, 56, 96, 103,
       95, 101, 123, 125, 91, 98, 85, 78, 0, 41, 92, 117, 82, 110, 108,
       74, 112, 90, 116, 151, 86, 45, 113, 105, 109, 48, 33, 44, 20, 37,
       134, 46, 126, 49, 39, 38, 36, 31, 25, 32, 17, 34, 27, 43, 29, 28,
       40, 30, 42, 100, 26, 129, 111, 120, 104, 119, 114, 124, 132, 121,
       122, 141, 161, 153, 130, 145, 156, 174, 138, 150, 140, 147, 148,
       131, 154, 127, 157, '65', '34', 'PM25', '52', '57', '81', '60',
       '50', '59', '43', '46', '61', '75', '74', '71', '63', '85', '77',
       '53', '68', '54', '55', '88', '58', '51', '62', '86', '0', '64',
       '96', '67', '94', '99', '56', '70', '66', '37', '91', '44', '80',
       '90', '72', '82', '78', '116', '100', '83', '179', '76', '73',
       '124', '136', '69', '102', '98', '121', '89', '42', 

In [73]:
data[data['max'] == 'PM25']

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
1677,2021-12-03,DKI1 (Bunderan HI),49,31,9,19,7,49,PM25,BAIK,NaN


In [74]:
# replace categori value from critical
data.loc[1677, 'categori'] = 'BAIK'

# replace critical value from max
data.loc[1677, 'critical'] = 'PM10'

# set max value
data.loc[1677, 'max'] = data.loc[1677, 'pm10']

In [75]:
data.loc[1677]

tanggal     2021-12-03 00:00:00
stasiun      DKI1 (Bunderan HI)
pm10                         49
pm25                         31
so2                           9
co                           19
o3                            7
no2                          49
max                          49
critical                   PM10
categori                   BAIK
Name: 1677, dtype: object

In [76]:
data['max'] = data['max'].astype(int)
print('max col data type:', data['max'].dtypes)

max col data type: int64


## Handling column categori

In [77]:
data['categori'].value_counts()

categori
SEDANG            1305
TIDAK SEHAT        319
BAIK               189
TIDAK ADA DATA      17
Name: count, dtype: int64

In [78]:
# drop the missing data
dropped_data = data[data['categori'] == 'TIDAK ADA DATA']
data.drop(index=dropped_data.index, inplace=True)

# sanity check categori values
data['categori'].value_counts()

categori
SEDANG         1305
TIDAK SEHAT     319
BAIK            189
Name: count, dtype: int64

In [79]:
# rename categori
data = data.rename(columns={'categori': 'category'})
data.head()

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,category
0,2021-03-01,DKI1 (Bunderan HI),37,60,20,12,12,17,60,PM25,SEDANG
1,2021-03-02,DKI1 (Bunderan HI),38,55,20,15,14,28,55,PM25,SEDANG
2,2021-03-03,DKI1 (Bunderan HI),49,67,19,21,17,35,67,PM25,SEDANG
3,2021-03-04,DKI1 (Bunderan HI),51,75,22,18,19,31,75,PM25,SEDANG
4,2021-03-05,DKI1 (Bunderan HI),52,69,20,16,16,30,69,PM25,SEDANG


In [80]:
# update the label in config

config = update_config(
    key='label',
    value='category',
    config=config,
    path_config='../config/config.yaml'
)

Config params updated! 
Key: label 
Value: category


In [81]:
# update the object column in config
col_object = config['object_columns']
col_object[-1] = 'category'

config = update_config(
    key='object_columns',
    value=col_object,
    config=config,
    path_config='../config/config.yaml'
)

Config params updated! 
Key: object_columns 
Value: ['stasiun', 'critical', 'category']


In [82]:
data.info()

<class 'pandas.DataFrame'>
Index: 1813 entries, 0 to 1829
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tanggal   1813 non-null   datetime64[us]
 1   stasiun   1813 non-null   str           
 2   pm10      1813 non-null   int64         
 3   pm25      1813 non-null   int64         
 4   so2       1813 non-null   int64         
 5   co        1813 non-null   int64         
 6   o3        1813 non-null   int64         
 7   no2       1813 non-null   int64         
 8   max       1813 non-null   int64         
 9   critical  1813 non-null   str           
 10  category  1813 non-null   str           
dtypes: datetime64[us](1), int64(7), str(3)
memory usage: 170.0 KB


In [83]:
# save the serialized data
joblib.dump(data, '../data/interim/validated_data.pkl')

['../data/interim/validated_data.pkl']

# **4. Update the Range of Data in Configuration File**

In [84]:
# update range of each numeric column on config
columns = list(data.select_dtypes(include='number').columns)
keys = ['pm10_range', 'pm25_range', 'so2_range', 'co_range', 'o3_range', 'no2_range']

for col, key in zip(columns, keys):
    config = update_config(
        key=key,
        value=[int(np.min(data[col])), int(np.max(data[col]))],
        config=config,
        path_config='../config/config.yaml'
    )

Config params updated! 
Key: pm10_range 
Value: [-1, 179]
Config params updated! 
Key: pm25_range 
Value: [-1, 174]
Config params updated! 
Key: so2_range 
Value: [-1, 82]
Config params updated! 
Key: co_range 
Value: [-1, 47]
Config params updated! 
Key: o3_range 
Value: [-1, 151]
Config params updated! 
Key: no2_range 
Value: [-1, 65]


In [85]:
# sanity check on config file

config

{'co_range': [-1, 47],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'no2_range': [-1, 65],
 'o3_range': [-1, 151],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'pm10_range': [-1, 179],
 'pm25_range': [-1, 174],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat'],
 'raw_data_path': '../data/raw/',
 'so2_range': [-1, 82]}

# **5. Data Defense**

In [86]:
def check_data(input_data, config):

    # check data types
    assert input_data.select_dtypes('datetime').columns.to_list() == config['datetime_columns'], 'an error occurs in datetime column(s).' 
    assert input_data.select_dtypes("str").columns.to_list() == config["object_columns"], "an error occurs in object column(s)."
    assert input_data.select_dtypes("number").columns.to_list() == config["int32_columns"], "an error occurs in int32 column(s)."    

    # check range of data
    assert set(input_data['stasiun']).issubset(set(config['range_stasiun'])), 'an error occurs in stasiun range.'
    assert input_data['pm10'].between(config['pm10_range'][0], config['pm10_range'][1]).sum() == len(input_data), "an error occurs in pm10 range."
    assert input_data['pm25'].between(config['pm25_range'][0], config['pm25_range'][1]).sum() == len(input_data), "an error occurs in pm25 range."
    assert input_data['so2'].between(config['so2_range'][0], config['so2_range'][1]).sum() == len(input_data), "an error occurs in so2 range."
    assert input_data['co'].between(config['co_range'][0], config['co_range'][1]).sum() == len(input_data), "an error occurs in co range."
    assert input_data['o3'].between(config['o3_range'][0], config['o3_range'][1]).sum() == len(input_data), "an error occurs in o3 range."
    assert input_data['no2'].between(config['no2_range'][0], config['no2_range'][1]).sum() == len(input_data), "an error occurs in no2 range."

In [87]:
check_data(input_data=data, config=config)

# **6. Data Split**

In [88]:
# Input-Output Split.
def split_input_output(input_data, params):
    """
    Split the input(X) and output (y).

    Parameters:
    ----------
    input_data : pd.DataFrame
        The processed dataset.

    params : dict
        Loaded configuration parameters.

    Returns:
    -------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.
    """

    X = input_data[params["features"]].copy()
    y = input_data[params["label"]].copy()

    print(f"Original data shape : {input_data.shape}")
    print(f"Selected Features   : {params["features"]}")
    print(f"X data shape        : {X.shape}")
    print(f"y data shape        : {y.shape}")

    return X, y

In [89]:
X, y = split_input_output(
    input_data = data,
    params = config
)

Original data shape : (1813, 11)
Selected Features   : ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
X data shape        : (1813, 7)
y data shape        : (1813,)


In [90]:
# Sanity check the input (X).
X.head()

,stasiun,pm10,pm25,so2,co,o3,no2
0,DKI1 (Bunderan HI),37,60,20,12,12,17
1,DKI1 (Bunderan HI),38,55,20,15,14,28
2,DKI1 (Bunderan HI),49,67,19,21,17,35
3,DKI1 (Bunderan HI),51,75,22,18,19,31
4,DKI1 (Bunderan HI),52,69,20,16,16,30


In [91]:
y.head()

0    SEDANG
1    SEDANG
2    SEDANG
3    SEDANG
4    SEDANG
Name: category, dtype: str

In [92]:
# Train-Test Split.
def split_train_test(X, y, test_size, random_state):
    """
    Split the train and test set.

    Parameters:
    ----------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.

    test_size : float
        The proportion of test set.

    random_state : int
        For reproducibility

    Returns:
    -------
    X_train, X_test : pd.DataFrame
        The train and test input.

    y_train, y_test : pd.Series
        The train and test output.
    """

    X_train, X_test, y_train, y_test = train_test_split(
                                            X, y,
                                            test_size = test_size,
                                            random_state = random_state,
                                            stratify = y
                                       )

    print(f"X_train shape : {X_train.shape}")
    print(f"y_train shape : {y_train.shape}")
    print(f"X_test shape  : {X_test.shape}")
    print(f"y_test shape  : {y_test.shape}")

    return X_train, X_test, y_train, y_test

In [93]:
RANDOM_STATE = 123

# Train vs not-train.
X_train, X_not_train, y_train, y_not_train = split_train_test(
    X = X,
    y = y,
    test_size = 0.2,
    random_state = RANDOM_STATE
)

print()
# Valid vs test.
X_valid, X_test, y_valid, y_test = split_train_test(
    X = X_not_train,
    y = y_not_train,
    test_size = 0.5,
    random_state = RANDOM_STATE
)

X_train shape : (1450, 7)
y_train shape : (1450,)
X_test shape  : (363, 7)
y_test shape  : (363,)

X_train shape : (181, 7)
y_train shape : (181,)
X_test shape  : (182, 7)
y_test shape  : (182,)


In [94]:
# Serialize the splitted data.
PATH_SPLITTED_DATA = f"../data/interim/"

joblib.dump(X_train, f"{PATH_SPLITTED_DATA}X_train.pkl")
joblib.dump(y_train, f"{PATH_SPLITTED_DATA}y_train.pkl")
joblib.dump(X_valid, f"{PATH_SPLITTED_DATA}X_valid.pkl")
joblib.dump(y_valid, f"{PATH_SPLITTED_DATA}y_valid.pkl")
joblib.dump(X_test, f"{PATH_SPLITTED_DATA}X_test.pkl")
joblib.dump(y_test, f"{PATH_SPLITTED_DATA}y_test.pkl")

['../data/interim/y_test.pkl']

In [95]:
# Update the configuration parameters.
config = update_config(
    key = "path_train_set",
    value = [f"{PATH_SPLITTED_DATA}X_train.pkl", f"{PATH_SPLITTED_DATA}y_train.pkl"],
    config = config,
    path_config = '../config/config.yaml'
)

config = update_config(
    key = "path_valid_set",
    value = [f"{PATH_SPLITTED_DATA}X_valid.pkl", f"{PATH_SPLITTED_DATA}y_valid.pkl"],
    config = config,
    path_config = '../config/config.yaml'
)

config = update_config(
    key = "path_test_set",
    value = [f"{PATH_SPLITTED_DATA}X_test.pkl", f"{PATH_SPLITTED_DATA}y_test.pkl"],
    config = config,
    path_config = '../config/config.yaml'
)

Config params updated! 
Key: path_train_set 
Value: ['../data/interim/X_train.pkl', '../data/interim/y_train.pkl']
Config params updated! 
Key: path_valid_set 
Value: ['../data/interim/X_valid.pkl', '../data/interim/y_valid.pkl']
Config params updated! 
Key: path_test_set 
Value: ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl']


In [96]:
# Check the configuration parameters.
config

{'co_range': [-1, 47],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'no2_range': [-1, 65],
 'o3_range': [-1, 151],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_test_set': ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl'],
 'path_train_set': ['../data/interim/X_train.pkl',
  '../data/interim/y_train.pkl'],
 'path_valid_set': ['../data/interim/X_valid.pkl',
  '../data/interim/y_valid.pkl'],
 'pm10_range': [-1, 179],
 'pm25_range': [-1, 174],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat'],
 'raw_data_path': '../data/raw/',
 'so2_range': [-1, 82]}